# Prédire le QIT (WPPSI-III) à partir de la HRV néonatale

*Groupe A · Jade · PRI 2026 · cohorte AuBE (CHU Saint-Étienne)*

**Question** : les indices de variabilité cardiaque (HRV) enregistrés à la naissance aident-ils à prédire le QI mesuré vers 3 ans, au-delà des facteurs cliniques ? Point de départ : dans le rapport de Samuel Fornet, les meilleurs modèles obtiennent une MAE d'environ 10,3 points et un R² ≤ 0, soit pas mieux que prédire le QIT moyen.

| Plan de la semaine | Section |
|---|---|
| **Mardi** · un enregistrement M0 par enfant, scores complets, fusion, nettoyage, jeu de données sauvegardé | 1 |
| **Mardi** · corrélation > 0,95, variance quasi nulle, sélection par forêt / Lasso / PCA, familles d'indices | 2 |
| **Mercredi** · Elastic Net, Random Forest, LightGBM, XGBoost, SVR et leurs espaces d'hyperparamètres | 3 |
| **Jeudi** · 10 × 5 validation croisée imbriquée, QIT / QIV / QIP / NCL, MAE, MSE, RMSE, R², `model_comparison.csv`, graphiques | 4 |
| **Vendredi** · analyse d'erreurs, comparaison à la moyenne, résumé pour le tuteur (`SUMMARY.md`), vérification des données | 4.4, 7, 8 |
| **Conseils du prof** · prédiction conforme, explication des variables | 5, 6 |

**Lancer** : `RAPIDE = True` pour vérifier que tout tourne (1 répétition, environ 7 minutes), puis `RAPIDE = False` pour les résultats à rapporter (10 répétitions, compter environ une heure). Le code technique est replié : cliquer dessus pour l'afficher. Les messages de vérification restent visibles. Les colonnes des fichiers sont décrites dans `GroupA/dataset.md`.

> **Données de santé d'enfants.** Les données et tout ce qui en est tiré au niveau individuel restent dans `data/` (ignoré par git). Le filtre `nbstripout` du dépôt vide les sorties du notebook à chaque commit. Ne jamais donner ces données à un outil d'IA.

In [ ]:
import hashlib
import re
import subprocess
import sys
import time
import unicodedata
import warnings
from datetime import date
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from lightgbm import LGBMRegressor
from scipy.stats import kruskal, rankdata
from xgboost import XGBRegressor

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, QuantileTransformer, StandardScaler
from sklearn.svm import SVR

# Racine du dépôt : le premier dossier qui contient evaluate_model.py, en remontant depuis celui du notebook.
RACINE = next((d for d in [Path.cwd(), *Path.cwd().parents] if (d / "evaluate_model.py").exists()), None)
if RACINE is None:
    raise FileNotFoundError(f"evaluate_model.py introuvable en remontant depuis {Path.cwd()} : "
                            "ouvrir le notebook depuis le dépôt FSIQ-from-ECG.")
sys.path.insert(0, str(RACINE))
from evaluate_model import evaluate, load_labels, norm_id, run_cv   # évaluation commune de l'équipe

In [ ]:
# ---------------------------------------------------------------- Réglages principaux
RAPIDE = False                       # True : vérification rapide ; False : résultats à rapporter
CIBLES = ["QIT", "QIV", "QIP", "NCL"]   # QIT = cible principale ; retirer les autres pour aller plus vite
T_REPETITIONS, K_PLIS = 10, 5       # 10 répétitions × 5 plis = 50 mesures par modèle et par cible
K_INTERNE = 5                       # plis de la boucle interne (réglage des hyperparamètres)
N_ITER = 4 if RAPIDE else 12        # combinaisons d'hyperparamètres tirées par réglage (RandomizedSearchCV)
ALPHA = 0.10                        # prédiction conforme : intervalle à 1 − α = 90 %

# Feature selection (section 2)
SEUIL_CORR = 0.95                   # [PISTE] 0.90 puis 0.80 : moins de variables redondantes
SEUIL_QUASI_CONSTANTE = 0.95        # variable retirée si une même valeur concerne 95 % des enfants
TOP_K = 30                          # [PISTE] variables gardées par la forêt ou le Lasso (15, 30, 50)
VARIANCE_PCA = 0.90                 # [PISTE] part de variance gardée par la PCA (0.80, 0.95)
PERIODES = ("24h", "Day", "Night")  # [PISTE] ("Night",) : sommeil seul ; ("24h",) : trois fois moins de variables
AJOUTER_NUIT_MOINS_JOUR = False     # [PISTE] ajoute Nuit − Jour pour chaque indice (réactivité circadienne)

REPETITIONS = [1] if RAPIDE else None   # None = les 10 répétitions de folds.csv
N_PERMUTATIONS = 20 if RAPIDE else 100
GRAINE, N_JOBS = 42, -1             # N_JOBS = -1 : tous les cœurs du processeur

In [ ]:
# ---------------------------------------------------------------- Fichiers
DOSSIER_DONNEES = RACINE / "data"
DOSSIER_PARTAGE = DOSSIER_DONNEES / "cleaned"   # cohorte, plis, prédictions, jeu nettoyé, modèles : local seulement
DOSSIER_RESULTATS = Path.cwd() / "results"      # model_comparison.csv, SUMMARY.md, graphique : chiffres agrégés
FICHIER_HRV_LIN, FICHIER_HRV_NL = "HRV_Lineaire_Full.csv", "HRV_NonLineaire_Full.csv"
FICHIER_WPPSI = "Résultats WPPSI III.csv"
REGENERER_PLIS = False              # [À ADAPTER] True pour recréer cohort.csv / folds.csv venus d'ailleurs

# ---------------------------------------------------------------- Colonnes
# La recherche ignore majuscules, accents et espaces ; un accent illisible (« Num�ro ») vaut n'importe quelle lettre.
COL_ID_HRV, COL_ID_WPPSI, COL_PASSAGE = "Numéro", "N°", "Passage"
PASSAGE_RETENU = "M0"               # [PISTE] M6, M12... : la HRV plus tardive prédit-elle mieux ?
LIGNES_ENTETE_WPPSI = 2             # « Note composite » au-dessus de « QIT » -> « Note composite | QIT »
LIGNES_ENTETE_HRV = 1
COLS_SCORES = {e: f"Note composite | {e}" for e in ["QIV", "QIP", "QIT", "NCL"]}
COLS_CLINIQUES = ["Sexe", "Prématuré", "SFA", "Réa à la naissance", "Pathologie neurologique",
                  "Tabac", "Diabète", "HTA", "Maturation Pulm", "Dépression: prise d'IRS"]
COLS_RR_ANALYSES, COL_RR_TOTAL = ["nb RR analyzed total", "nb analyzed RR total"], "nb RR total"
QUALITE_MIN = None                  # [PISTE] ex. 70 : écarte les enregistrements où moins de 70 % des RR sont analysés
# Jamais en variables d'entrée : identité, dates, horaires, durées, nombres de RR, scores WPPSI.
MOTIFS_EXCLUS = [r"numero", r"^n°", r"monogramme", r"ddn", r"filename", r"fichier", r"recorder", r"date",
                 r"heure", r"duree", r"duration", r"beginning", r"\b(debut|fin|start|end)\b", r"nb ?rr",
                 r"analy[sz]ed", r"passage", r"sortie", r"^(qit|qiv|qip|ncl|com|inf|dim|cub|aob)\b"]

# ---------------------------------------------------------------- Nettoyage et prétraitement
MAX_MANQUANTS_COL, MAX_MANQUANTS_LIGNE = 0.30, 0.50
MIN_CAS_RARES = 5                   # covariable binaire retirée si une classe a moins de 5 enfants
TRANSFO = "quantile"                # [PISTE] "yeo-johnson" (≈ 20 fois plus lent) ou "standard" (StandardScaler)

# ---------------------------------------------------------------- Elastic Net
# sklearn : `alpha` = force de la pénalité (λ dans glmnet), `l1_ratio` = mélange L1/L2 (α dans glmnet).
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]   # plan du mercredi ; [PISTE] ajouter 0.05 et 1.0
N_ALPHAS, EPS_ALPHA = 30, 1e-2      # 30 valeurs d'alpha, de alpha_max à alpha_max / 100

# 1. Preprocessing

## 1.1 Lecture des fichiers

Le contenu des trois fichiers est décrit dans `GroupA/dataset.md`. Ce sont des exports Excel : encodage, séparateur, virgule décimale, ligne d'unités sous l'en-tête, cases « - » et accents illisibles sont détectés automatiquement.

In [ ]:
def normaliser_texte(s):
    """Minuscules, sans accents, espaces simplifiés : ' Numéro ' -> 'numero'."""
    s = unicodedata.normalize("NFKD", str(s))
    return re.sub(r"\s+", " ", "".join(c for c in s if not unicodedata.combining(c))).strip().lower()


def cle(s):
    return re.sub(r"[\s_.\-]", "", normaliser_texte(s))


def meme_nom(colonne, nom):
    """Compare les clés ; un caractère illisible « � » dans le nom de colonne vaut n'importe quel caractère."""
    motif = "".join("." if ch == "\ufffd" else re.escape(ch) for ch in cle(colonne))
    return re.fullmatch(motif, cle(nom)) is not None


def trouver_colonne(df, *noms):
    """Vrai nom de la colonne qui correspond à l'un des noms (ou à la fin d'un nom « Bloc | Nom »), sinon None."""
    for nom in noms:
        exactes = [c for c in df.columns if meme_nom(c, nom)]
        if exactes:
            return exactes[0]
        fins = [c for c in df.columns if meme_nom(str(c).split("|")[-1], nom)]
        if len(fins) == 1:
            return fins[0]
    return None


def aplatir_entetes(colonnes):
    """Deux lignes d'en-tête (cellules fusionnées d'Excel) -> « Bloc | Sous-colonne »."""
    noms, bloc = [], ""
    for haut, bas in colonnes:
        haut = "" if str(haut).startswith("Unnamed") else str(haut).strip()
        bas = "" if str(bas).startswith("Unnamed") else str(bas).strip()
        bloc = haut or bloc        # une cellule fusionnée n'est remplie que sur sa première colonne
        noms.append(f"{bloc} | {bas}" if bloc and bas else bloc or bas)
    return noms


LETTRES_FR = set("éèêëàâäùûüçîïôö°")
UNITES = {"-", "n", "ms", "ms^2", "ms2", "ms/beat", "bpm", "%", "hz", "s", "hh:mm:ss", "u.a.", "n.u.", "0/1", "m/f", "unit"}
VALEURS_VIDES = {"", "-", "--", "na", "n/a", "nan", "nd", "?", "#n/a", "#div/0!", "#valeur!", "#value!", "#ref!", "#nombre!"}


def detecter_encodage(brut):
    """UTF-16, UTF-8, Windows (cp1252) ou Excel Mac (mac_roman)."""
    if brut[:2] in (b"\xff\xfe", b"\xfe\xff"):
        return "utf-16"
    try:
        brut.decode("utf-8-sig")
        return "utf-8-sig"
    except UnicodeDecodeError:
        pass
    return max(["cp1252", "mac_roman"], key=lambda e: sum(ch in LETTRES_FR for ch in brut.decode(e, errors="replace")))


def reparer_accents(nom):
    """'NumÃ©ro' (UTF-8 relu comme Windows) -> 'Numéro'."""
    try:
        return nom.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return nom


def lire_csv(chemin, lignes_entete=1):
    brut = Path(chemin).read_bytes()
    encodage = detecter_encodage(brut)
    lignes = [l for l in brut.decode(encodage, errors="replace").splitlines() if l.strip()]
    a_sauter = 1 if lignes[0].strip().lower().startswith("sep=") else 0   # ligne ajoutée par certains exports Excel
    sep = max([";", ",", "\t"], key=lignes[a_sauter].count)
    df = pd.read_csv(chemin, sep=sep, encoding=encodage, encoding_errors="replace", skiprows=a_sauter,
                     header=list(range(lignes_entete)), decimal="," if sep != "," else ".")
    if lignes_entete > 1:
        df.columns = aplatir_entetes(df.columns)
    noms, vus = [], {}
    for c in df.columns:
        nom = reparer_accents(str(c).strip()) or "sans_nom"
        vus[nom] = vus.get(nom, 0) + 1
        noms.append(nom if vus[nom] == 1 else f"{nom}.{vus[nom] - 1}")
    df.columns = noms
    lecture = f"encodage {encodage}, séparateur {sep!r}" + (", ligne « sep= » ignorée" if a_sauter else "")
    ligne_1 = df.iloc[0].astype(str).tolist() if len(df) else []
    # Ligne d'unités / légendes sous l'en-tête (M/F, 0/1, ms, bpm, %...) : ce ne sont pas des données.
    valeurs = [normaliser_texte(v) for v in (df.iloc[0] if len(df) else []) if pd.notna(v) and str(v).strip()]
    if valeurs and sum(v in UNITES for v in valeurs) >= 0.5 * len(valeurs):
        df = df.iloc[1:].reset_index(drop=True)
        lecture += ", ligne d'unités ignorée"
    df.attrs["lecture"], df.attrs["ligne_1"] = lecture, ligne_1
    # Colonnes numériques restées en texte (virgule décimale, %, « - » pour une case vide...) :
    # converties si au moins 90 % des valeurs renseignées sont des nombres.
    presque = {}
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any():
            texte = df[c].astype("string").str.replace(r"[\s\u00a0\u202f%]", "", regex=True).str.replace(",", ".")
            texte = texte.where(~texte.str.lower().isin(VALEURS_VIDES).fillna(False).astype(bool))
            nombres = pd.to_numeric(texte, errors="coerce").astype("float64")
            n_remplies, n_nombres = texte.notna().sum(), nombres.notna().sum()
            if n_nombres >= 0.9 * n_remplies:
                df[c] = nombres
            elif n_nombres >= 0.5 * n_remplies:
                presque[c] = (f"{n_nombres / n_remplies:.0%} de nombres",
                              texte[nombres.isna() & texte.notna()].str.slice(0, 12).value_counts().head(3).index.tolist())
    df.attrs["presque_numeriques"] = presque
    return df


hrv_lin_brut = lire_csv(DOSSIER_DONNEES / FICHIER_HRV_LIN, LIGNES_ENTETE_HRV)
hrv_nl_brut = lire_csv(DOSSIER_DONNEES / FICHIER_HRV_NL, LIGNES_ENTETE_HRV)
wppsi_brut = lire_csv(DOSSIER_DONNEES / FICHIER_WPPSI, LIGNES_ENTETE_WPPSI)

for nom, df in [("HRV linéaire", hrv_lin_brut), ("HRV non linéaire", hrv_nl_brut), ("WPPSI", wppsi_brut)]:
    n_num = sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)
    print(f"{nom:17s}: {df.shape[0]} lignes, {df.shape[1]} colonnes dont {n_num} numériques ({df.attrs['lecture']})")
    if df.attrs["presque_numeriques"]:
        # [À ADAPTER] ajouter ces valeurs à VALEURS_VIDES si ce sont des marqueurs de case vide
        print(f"   presque numériques mais restées en texte : {dict(list(df.attrs['presque_numeriques'].items())[:10])}")
print("\nColonnes WPPSI :", list(wppsi_brut.columns))

## 1.2 Identifiants

Les fichiers sont reliés par le **numéro** du sujet (1re colonne), harmonisé comme dans `evaluate_model.py` (« 012 », « 12.0 » → « 12 »). Le plan proposait Monogramme + DDN, mais des initiales se répètent d'un enfant à l'autre (voir `data/pulisci_righe.py`). La date de naissance sert donc de vérification (1.5). Seuls les formats des identifiants sont affichés (`A` = lettre, `9` = chiffre).

In [ ]:
MOTIF_ID = re.compile(r"^((n°|no|num|numero|id|identifiant|code)(patient|sujet|enfant|aube|dossier)?|patient|sujet|enfant)$")


def trouver_id(df, nom_configure):
    col = trouver_colonne(df, nom_configure)
    return col if col is not None else next((c for c in df.columns if MOTIF_ID.match(cle(c))), None)


fichiers_bruts = {"lin": hrv_lin_brut, "nl": hrv_nl_brut, "wppsi": wppsi_brut}
col_id = {nom: trouver_id(df, COL_ID_WPPSI if nom == "wppsi" else COL_ID_HRV) for nom, df in fichiers_bruts.items()}
if None in col_id.values():
    for nom, df in fichiers_bruts.items():
        if col_id[nom] is None:
            print(f"{nom} ({df.attrs['lecture']}) : {list(df.columns[:20])}")
            if any(MOTIF_ID.match(cle(v)) for v in df.attrs["ligne_1"]):
                print("  -> l'identifiant est sur la 1re ligne de données : en-tête sur 2 lignes (LIGNES_ENTETE_... = 2)")
    raise KeyError("Colonne identifiant introuvable : mettre son nom exact dans COL_ID_HRV / COL_ID_WPPSI.")

for nom, df in fichiers_bruts.items():
    df["id_enfant"] = df[col_id[nom]].map(norm_id)
    formats = df[col_id[nom]].dropna().astype(str).str.strip().map(lambda s: re.sub(r"\d", "9", re.sub(r"[A-Za-z]", "A", s)))
    print(f"{nom:6s} colonne « {col_id[nom]} » : {df['id_enfant'].nunique()} identifiants, "
          f"formats {formats.value_counts().head(3).to_dict()}")
communs = set.intersection(*(set(df["id_enfant"].dropna()) for df in fichiers_bruts.values()))
print(f"Identifiants présents dans les 3 fichiers : {len(communs)}")

## 1.3 Scores WPPSI

QIV, QIP, QIT et NCL (note composite de langage) sont les cibles possibles ; ils ne servent jamais de variables d'entrée, pas plus que les sous-tests. Comme dans `data/pulisci_righe.py`, un enfant sans tous ses scores est retiré.

In [ ]:
wppsi = wppsi_brut.dropna(subset=["id_enfant"])


def trouver_score(echelle, nom_configure):
    """Colonne de la note composite d'une échelle ; à défaut, colonne au nom de l'échelle dont la médiane ressemble à un QI."""
    col = trouver_colonne(wppsi, nom_configure)
    if col is not None and pd.api.types.is_numeric_dtype(wppsi[col]):
        return col
    candidates = [c for c in wppsi.columns if echelle.lower() in normaliser_texte(c) and pd.api.types.is_numeric_dtype(wppsi[c])]
    plausibles = [c for c in candidates if 70 <= wppsi[c].median() <= 130]   # le rang percentile tourne autour de 50
    return plausibles[0] if len(plausibles) == 1 else None


cols_scores = {e: c for e, nom in COLS_SCORES.items() if (c := trouver_score(e, nom)) is not None}
if "QIT" not in cols_scores:
    raise ValueError("QIT introuvable : corriger COLS_SCORES ou LIGNES_ENTETE_WPPSI.")
absentes = [c for c in CIBLES if c not in cols_scores]
if absentes:
    print(f"Scores introuvables, retirés des cibles : {absentes}")   # [À ADAPTER]
CIBLES = [c for c in CIBLES if c in cols_scores]

# Plusieurs lignes pour un enfant : 1re valeur non vide ; deux QIT différents -> enfant retiré.
scores = wppsi[["id_enfant", *cols_scores.values()]].rename(columns={c: e for e, c in cols_scores.items()})
n_qit = scores.dropna(subset=["QIT"]).groupby("id_enfant")["QIT"].nunique()
conflits = n_qit.index[n_qit > 1]
scores = scores[~scores["id_enfant"].isin(conflits)].groupby("id_enfant").first()
incomplets = scores.isna().any(axis=1)
hors_bornes = ~incomplets & ~scores.apply(lambda s: s.between(40, 160)).all(axis=1)
scores = scores[~incomplets & ~hors_bornes]
print(f"{len(conflits)} enfants aux QIT contradictoires, {incomplets.sum()} sans tous les scores, "
      f"{hors_bornes.sum()} avec un score hors de [40, 160] : retirés")
display(scores.describe().loc[["count", "mean", "std", "min", "max"]].round(1))   # [À ADAPTER] médianes proches de 100 ?

## 1.4 Indices HRV : un enregistrement M0 par enfant

Le passage M0 est parfois écrit « MO » (lettre O) : les deux sont acceptés. Si un enfant a deux enregistrements M0, on garde le mieux analysé (part de RR analysés). Les colonnes d'identité, de dates, de durées et de nombres de RR sont exclues des variables ; la liste est affichée pour vérification.

In [ ]:
def normaliser_passage(valeur):
    """'MO' (lettre O), 'm0', 'M00' -> 'M0' ; 'M06' -> 'M6'."""
    s = re.sub(r"\s", "", str(valeur)).upper()
    return re.sub(r"^M0+(?=\d)", "M", "M0" if s == "MO" else s)


def periode(colonne):
    """'SDNN (TDNight)' -> 'Night', 'Higuchi (24h)' -> '24h', sans période -> None."""
    m = re.search(r"(24h|Day|Night)\)$", colonne)
    return m.group(1) if m else None


def preparer_hrv(brut, bloc):
    df = brut.dropna(subset=["id_enfant"])
    col_passage = trouver_colonne(df, COL_PASSAGE)
    if col_passage is not None:
        passage = df[col_passage].map(normaliser_passage)
        print(f"[{bloc}] passages : {passage.value_counts().to_dict()}")
        df = df[passage == normaliser_passage(PASSAGE_RETENU)]

    col_ana, col_tot = trouver_colonne(df, *COLS_RR_ANALYSES), trouver_colonne(df, COL_RR_TOTAL)
    if col_ana is not None and col_tot is not None and all(pd.api.types.is_numeric_dtype(df[c]) for c in (col_ana, col_tot)):
        qualite = 100 * df[col_ana] / df[col_tot]
        print(f"[{bloc}] RR analysés : médiane {qualite.median():.0f} %, minimum {qualite.min():.0f} %")
        if QUALITE_MIN is not None:
            df, qualite = df[qualite >= QUALITE_MIN], qualite[qualite >= QUALITE_MIN]
        df = df.loc[qualite.sort_values(ascending=False).index]
    n_doublons = df["id_enfant"].duplicated().sum()
    df = df.drop_duplicates("id_enfant").set_index("id_enfant")

    cliniques = [c for c in (trouver_colonne(df, n) for n in COLS_CLINIQUES) if c is not None]
    exclues = [c for c in df.columns
               if c in cliniques or c == col_id[bloc] or any(re.search(m, normaliser_texte(c)) for m in MOTIFS_EXCLUS)]
    indices = [c for c in df.columns if c not in exclues and pd.api.types.is_numeric_dtype(df[c])]
    meta = df.drop(columns=indices)
    indices = [c for c in indices if periode(c) in (None, *PERIODES)]
    hrv = df[indices]
    if AJOUTER_NUIT_MOINS_JOUR:
        nuit_jour = {c[:-4] + "Nuit-Jour)": df[c[:-4] + "Night)"] - df[c]
                     for c in indices if c.endswith("Day)") and c[:-4] + "Night)" in indices}
        hrv = pd.concat([hrv, pd.DataFrame(nuit_jour)], axis=1)
    print(f"[{bloc}] {len(df)} enfants au passage {PASSAGE_RETENU} ({n_doublons} doublons écartés), "
          f"{hrv.shape[1]} indices HRV gardés")
    print(f"[{bloc}] exclues des variables : {exclues}")   # [À ADAPTER] compléter MOTIFS_EXCLUS si besoin
    return hrv.add_prefix(f"{bloc}|"), meta


hrv_lin, meta_lin = preparer_hrv(hrv_lin_brut, "lin")
hrv_nl, meta_nl = preparer_hrv(hrv_nl_brut, "nl")

## 1.5 Fusion et nettoyage

On garde les enfants présents dans les 3 fichiers (décision de l'équipe : un enfant sans enregistrement M0 n'a pas les mêmes informations que les autres), dont la date de naissance concorde d'un fichier à l'autre. Les covariables cliniques sont ajoutées, puis on retire les variables trop vides, constantes ou quasi constantes. Aucune de ces étapes ne regarde le QIT, donc aucune information sur la cible ne fuit.

In [ ]:
donnees = hrv_lin.join(hrv_nl, how="inner").join(scores, how="inner")
print(f"Enfants : {len(hrv_lin)} (HRV linéaire), {len(hrv_nl)} (non linéaire), {len(scores)} (scores) "
      f"-> {len(donnees)} dans les 3 fichiers")

# Même numéro = même enfant ? Dates de naissance comparées entre les 3 fichiers.
wppsi_enfant = wppsi.groupby("id_enfant").first()


def dates_naissance(source):
    col = trouver_colonne(source, "DDN")
    if col is None:
        return None
    texte = source[col].astype("string").str.strip()
    return pd.to_datetime(texte, dayfirst=True, errors="coerce", format="mixed").reindex(donnees.index)


ddn = pd.DataFrame({nom: d for nom, source in [("lin", meta_lin), ("nl", meta_nl), ("wppsi", wppsi_enfant)]
                    if (d := dates_naissance(source)) is not None})
differentes = (ddn.notna().sum(axis=1) >= 2) & (ddn.nunique(axis=1) > 1)
donnees = donnees[~differentes]
print(f"{differentes.sum()} enfants dont la date de naissance diffère entre fichiers : retirés")   # [À ADAPTER] à vérifier

# Covariables cliniques (fichier HRV linéaire en priorité) ; une classe de moins de MIN_CAS_RARES enfants est inutilisable.
clinique = pd.DataFrame(index=donnees.index)
for nom in COLS_CLINIQUES:
    source = next((s for s in [meta_lin, meta_nl, wppsi_enfant] if trouver_colonne(s, nom) is not None), None)
    if source is not None:
        valeurs = source[trouver_colonne(source, nom)].reindex(donnees.index)
        clinique[f"clin|{nom}"] = valeurs if pd.api.types.is_numeric_dtype(valeurs) else valeurs.astype(object).where(valeurs.notna(), np.nan)
rares = [c for c in clinique.columns
         if clinique[c].nunique() < 2 or (clinique[c].nunique() == 2 and clinique[c].value_counts().min() < MIN_CAS_RARES)]
introuvables = [n for n in COLS_CLINIQUES if f"clin|{n}" not in clinique.columns]
donnees = donnees.join(clinique.drop(columns=rares))
print(f"Covariables cliniques : {clinique.shape[1] - len(rares)} gardées, trop rares {rares}, introuvables {introuvables}")

# Variables trop vides, constantes ou quasi constantes (une même valeur pour ≥ 95 % des enfants).
cols_hrv = list(hrv_lin.columns) + list(hrv_nl.columns)
donnees[cols_hrv] = donnees[cols_hrv].replace([np.inf, -np.inf], np.nan)
part_manquante = donnees[cols_hrv].isna().mean()
part_dominante = donnees[cols_hrv].apply(lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else 1.0)
a_retirer = part_manquante.index[(part_manquante > MAX_MANQUANTS_COL) | (part_dominante >= SEUIL_QUASI_CONSTANTE)]
donnees = donnees.drop(columns=a_retirer)
incomplets = donnees[[c for c in cols_hrv if c not in a_retirer]].isna().mean(axis=1) > MAX_MANQUANTS_LIGNE
donnees = donnees[~incomplets]
print(f"{len(a_retirer)} variables trop vides ou quasi constantes et {incomplets.sum()} enfants trop incomplets retirés")

## 1.6 Cohorte et plis communs, jeu de données sauvegardé

`evaluate_model.py` ne compare des modèles que s'ils ont prédit **les mêmes enfants avec les mêmes plis** : `cohort.csv` (child_id et scores) et `folds.csv` (child_id, repeat, fold). Ces fichiers devraient venir de `prepare_data.py` de l'équipe. En attendant, le notebook les crée au même format (10 répétitions de 5 plis stratifiés sur le QIT) et les recrée lui-même si son nettoyage change. Des fichiers fournis par l'équipe dans `data/cleaned/` sont utilisés tels quels. Le jeu de données nettoyé est sauvegardé à côté, localement (plan du mardi).

In [ ]:
fichier_cohorte, fichier_plis = DOSSIER_PARTAGE / "cohort.csv", DOSSIER_PARTAGE / "folds.csv"
marque = DOSSIER_PARTAGE / "cohort_folds.notebook"     # empreinte des fichiers créés par ce notebook


def empreinte():
    return "".join(hashlib.sha256(f.read_bytes()).hexdigest() for f in (fichier_cohorte, fichier_plis))


existent = fichier_cohorte.exists() and fichier_plis.exists()
crees_ici = existent and marque.exists() and marque.read_text() == empreinte()
perimes = crees_ici and set(pd.read_csv(fichier_cohorte, dtype={"child_id": str})["child_id"].map(norm_id)) != set(donnees.index)
if REGENERER_PLIS or not existent or perimes:
    DOSSIER_PARTAGE.mkdir(parents=True, exist_ok=True)
    donnees[list(scores.columns)].rename_axis("child_id").reset_index().to_csv(fichier_cohorte, index=False)
    tranches = pd.qcut(donnees["QIT"], q=5, labels=False, duplicates="drop")   # QIT bas, moyens et hauts dans chaque pli
    plis = [pd.DataFrame({"child_id": donnees.index[te], "repeat": r, "fold": k})
            for r in range(1, T_REPETITIONS + 1)
            for k, (_, te) in enumerate(StratifiedKFold(K_PLIS, shuffle=True, random_state=GRAINE + r)
                                        .split(donnees, tranches), start=1)]
    pd.concat(plis).to_csv(fichier_plis, index=False)
    marque.write_text(empreinte())
    print(f"cohort.csv et folds.csv créés : {len(donnees)} enfants, {T_REPETITIONS} × {K_PLIS} plis")
else:
    print("cohort.csv et folds.csv existants utilisés"
          + ("" if crees_ici else " (venus de l'équipe ou d'une version précédente du notebook)"))

plis_partages = pd.read_csv(fichier_plis, dtype={"child_id": str})
plis_partages["child_id"] = plis_partages["child_id"].map(norm_id)
absentes = [c for c in CIBLES if c not in pd.read_csv(fichier_cohorte, nrows=0).columns]
if absentes:
    print(f"Cibles absentes de cohort.csv, retirées : {absentes}. Si ce fichier vient d'un ancien passage du notebook, "
          "mettre REGENERER_PLIS = True pour le recréer avec toutes les cibles.")   # [À ADAPTER]
    CIBLES = [c for c in CIBLES if c not in absentes]
etiquettes = load_labels(fichier_cohorte, CIBLES)
sans_variables = set(plis_partages["child_id"]) - set(donnees.index)
if sans_variables:
    raise ValueError(f"{len(sans_variables)} enfants de la cohorte commune n'ont pas de variables ici : comparer les "
                     "nettoyages avec l'équipe, ou REGENERER_PLIS = True pour repartir de ce notebook.")
donnees = donnees.loc[sorted(plis_partages["child_id"].unique(), key=int)]
donnees.rename_axis("child_id").to_csv(DOSSIER_PARTAGE / "dataset_m0.csv")

X = donnees.drop(columns=list(scores.columns))
cols_lin = [c for c in X.columns if c.startswith("lin|")]
cols_nl = [c for c in X.columns if c.startswith("nl|")]
cols_clin = [c for c in X.columns if c.startswith("clin|")]
CLIN_TEXTE = {c for c in cols_clin if not pd.api.types.is_numeric_dtype(X[c])}
display(pd.DataFrame({"enfants": [len(X)], "indices linéaires": [len(cols_lin)], "indices non linéaires": [len(cols_nl)],
                      "covariables cliniques": [len(cols_clin)], "variables par enfant": [round(X.shape[1] / len(X), 1)]},
                     index=["jeu final"]))

## 1.7 Contrôles

- **Effet d'appareil** : les enregistrements viennent de deux appareils (EDF et Novacor Vista, d'après `dataset.md`). Si le QIT moyen diffère selon l'appareil, un modèle pourrait apprendre l'appareil plutôt que la physiologie.
- **Repère de signal** : la plus forte corrélation d'une variable avec le QIT, comparée à celle qu'on obtient par pur hasard avec autant de variables. C'est descriptif seulement : choisir des variables ici, hors des plis, serait une fuite d'information.

In [ ]:
col_appareil = trouver_colonne(meta_lin, "Recorder")
if col_appareil is not None:
    appareil = meta_lin[col_appareil].reindex(donnees.index)
    groupes = [g["QIT"].to_numpy() for _, g in donnees.groupby(appareil) if len(g) >= 5]
    p_appareil = kruskal(*groupes).pvalue if len(groupes) >= 2 else np.nan
    print(f"QIT moyen par appareil : {donnees.groupby(appareil)['QIT'].mean().round(1).to_dict()} "
          f"(Kruskal-Wallis p = {p_appareil:.3f})")   # [PISTE] si p < 0.05 : appareil en covariable

numeriques = X.select_dtypes("number")
rho = numeriques.corrwith(donnees["QIT"], method="spearman").abs().sort_values(ascending=False)
melange = np.random.default_rng(0)
rho_hasard = np.mean([numeriques.corrwith(pd.Series(melange.permutation(donnees["QIT"].to_numpy()), index=X.index),
                                          method="spearman").abs().max() for _ in range(20)])
print(f"|Spearman| le plus fort avec le QIT : {rho.iloc[0]:.2f}, contre {rho_hasard:.2f} en moyenne par pur hasard "
      f"avec {numeriques.shape[1]} variables. Les 5 plus fortes : {rho.head(5).round(2).to_dict()}")

# 2. Feature selection

Avec environ 200 variables pour 180 enfants, un modèle peut apprendre le bruit plutôt qu'un vrai lien. Le nombre de variables est donc réduit **dans chaque pli, sur les seuls enfants d'entraînement** ; une sélection faite sur tous les enfants avant la validation croisée donnerait des performances trop optimistes.

1. **Prétraitement** (toujours) : imputation par la médiane, une seule variable gardée par groupe corrélé à plus de 0,95, transformation qui réduit l'asymétrie et met les variables à la même échelle.
2. **Sélection**, au choix pour chaque expérience :

| Option | Principe |
|---|---|
| `toutes` | pas de sélection : le modèle fait le tri (coefficients nuls d'Elastic Net, coupures des arbres) |
| `sél. RF` | les `TOP_K` variables les plus importantes d'une forêt aléatoire de référence (conseil du prof) |
| `sél. Lasso` | les `TOP_K` plus gros coefficients d'un Lasso (criblage du plan) |
| `PCA 90 %` | les composantes principales qui gardent 90 % de la variance |

3. **Familles d'indices** (plan du mardi, `dataset.md`) : temporels (`TD`), fréquentiels (`FT`, et les indices en ondelettes de nuit `IncNight`), géométriques (`GI`), non linéaires. Elles servent à voir si une famille porte seule le signal.

In [ ]:
class FiltreCorrelation(BaseEstimator, TransformerMixin):
    """Garde une variable par groupe de variables très corrélées (|Spearman| > seuil), ajusté dans chaque pli."""

    def __init__(self, seuil=0.95):
        self.seuil = seuil

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        if self.seuil is None or X.shape[1] < 2:
            self.garder_ = np.arange(X.shape[1])
            return self
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.abs(np.nan_to_num(np.corrcoef(rankdata(X, axis=0), rowvar=False)))
        garder = []
        for j in range(X.shape[1]):
            if not garder or corr[j, garder].max() <= self.seuil:
                garder.append(j)
        self.garder_ = np.array(garder)
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.garder_]

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = [f"x{i}" for i in range(self.n_features_in_)]
        return np.asarray(input_features, dtype=object)[self.garder_]


def faire_transfo():
    if TRANSFO == "yeo-johnson":
        return PowerTransformer(method="yeo-johnson")
    if TRANSFO == "quantile":
        return QuantileTransformer(n_quantiles=50, output_distribution="normal")
    return StandardScaler()


def faire_preprocesseur(colonnes):
    hrv = [c for c in colonnes if not c.startswith("clin|")]
    clin_num = [c for c in colonnes if c.startswith("clin|") and c not in CLIN_TEXTE]
    clin_texte = [c for c in colonnes if c in CLIN_TEXTE]
    etapes = []
    if hrv:
        etapes.append(("hrv", Pipeline([("imputation", SimpleImputer(strategy="median")),
                                        ("correlation", FiltreCorrelation(SEUIL_CORR)),
                                        ("transfo", faire_transfo())]), hrv))
    if clin_num:
        etapes.append(("clin_num", Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                                            ("echelle", StandardScaler())]), clin_num))
    if clin_texte:
        etapes.append(("clin_texte", Pipeline([("imputation", SimpleImputer(strategy="most_frequent")),
                                              ("codage", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                                                                       sparse_output=False)),
                                              ("echelle", StandardScaler())]), clin_texte))
    return ColumnTransformer(etapes, verbose_feature_names_out=False)


def famille(colonne):
    if colonne.startswith("nl|"):
        return "non linéaire"
    if "(TD" in colonne:
        return "temporel"
    return "géométrique" if "(GI" in colonne else "fréquentiel"

In [ ]:
def top_k(X):
    return min(TOP_K, X.shape[1])


SELECTIONS = {
    "toutes": lambda: "passthrough",
    "sél. RF": lambda: SelectFromModel(RandomForestRegressor(n_estimators=200, max_features="sqrt", min_samples_leaf=5,
                                                             random_state=0, n_jobs=1),
                                       max_features=top_k, threshold=-np.inf),
    "sél. Lasso": lambda: SelectFromModel(LassoCV(alphas=20, cv=3, max_iter=5_000, selection="random", random_state=0),
                                          max_features=top_k, threshold=-np.inf),
    "PCA 90 %": lambda: PCA(n_components=VARIANCE_PCA, svd_solver="full"),
}

cols_hrv = cols_lin + cols_nl
JEUX = {"Clinique seul": cols_clin, "HRV seule": cols_hrv, "Clinique + HRV": cols_clin + cols_hrv}
for nom_famille in ["temporel", "fréquentiel", "géométrique", "non linéaire"]:
    JEUX[f"HRV {nom_famille}"] = [c for c in cols_hrv if famille(c) == nom_famille]
print({nom: len(cols) for nom, cols in JEUX.items()})

# 3. Models

Cinq modèles, chacun un estimateur scikit-learn (`fit`, `predict`, `get_params`, `set_params`). Leurs hyperparamètres sont réglés dans une **boucle interne à 5 plis** sur le pli d'entraînement, en minimisant la MAE :

- **Elastic Net** : `ElasticNetCV` parcourt tout le chemin de pénalité d'un coup, de `alpha_max` (aucune variable) à `alpha_max / 100` ;
- **les autres** : `RandomizedSearchCV` tire `N_ITER` combinaisons dans les espaces du plan du mercredi.

Pour aller vite, le prétraitement et la sélection sont ajustés une fois par pli externe, et la boucle interne ne règle que le modèle. Le pli de test externe n'est jamais vu, donc l'estimation reste honnête. Le prof conseille surtout **LightGBM** et **XGBoost**.

Une **expérience** = un modèle + une sélection + un jeu de variables. Toutes voient les mêmes enfants et les mêmes plis.

In [ ]:
INTERNE = KFold(K_INTERNE, shuffle=True, random_state=0)

# Espaces d'hyperparamètres du plan du mercredi.
ESPACES = {
    "Random Forest": {"n_estimators": [200, 500], "max_depth": [3, 5, 7, None], "min_samples_leaf": [5, 10, 20],
                      "max_features": ["sqrt", "log2"]},
    "LightGBM": {"n_estimators": [100, 300], "learning_rate": [0.01, 0.05, 0.1], "max_depth": [3, 5, -1],
                 "min_child_samples": [10, 20], "reg_alpha": [0, 0.1, 1], "reg_lambda": [0, 1, 10]},
    "XGBoost": {"n_estimators": [100, 300], "learning_rate": [0.01, 0.05, 0.1], "max_depth": [3, 5, 7],
                "min_child_weight": [1, 5], "reg_alpha": [0, 0.1, 1], "reg_lambda": [1, 10], "subsample": [0.8, 1.0]},
    "SVR": {"C": [0.1, 1, 10, 100], "epsilon": [0.01, 0.1, 1], "kernel": ["rbf", "linear"], "gamma": ["scale", "auto"]},
}
BASES = {
    "Random Forest": lambda: RandomForestRegressor(random_state=0, n_jobs=1),
    "LightGBM": lambda: LGBMRegressor(colsample_bytree=0.5, subsample=0.8, subsample_freq=1, random_state=0,
                                      n_jobs=1, verbose=-1),
    "XGBoost": lambda: XGBRegressor(colsample_bytree=0.5, random_state=0, n_jobs=1),
    "SVR": lambda: SVR(),
}


def faire_modele(nom):
    if nom == "Elastic Net":
        return ElasticNetCV(l1_ratio=L1_RATIOS, alphas=N_ALPHAS, eps=EPS_ALPHA, cv=INTERNE, max_iter=10_000,
                            selection="random", random_state=0)   # "random" : ≈ 6 fois plus rapide ici
    return RandomizedSearchCV(BASES[nom](), ESPACES[nom], n_iter=N_ITER, cv=INTERNE,
                              scoring="neg_mean_absolute_error", random_state=0)


def faire_pipeline(modele, selection, colonnes):
    return Pipeline([("prep", faire_preprocesseur(colonnes)), ("selection", SELECTIONS[selection]()),
                     ("modele", faire_modele(modele))])


EXPERIENCES = [                               # (modèle, sélection, jeu de variables)
    ("Elastic Net", "toutes", "Clinique seul"),     # référence clinique : ce que la HRV doit améliorer
    ("Elastic Net", "toutes", "Clinique + HRV"),
    ("Random Forest", "toutes", "Clinique + HRV"),
    ("LightGBM", "sél. RF", "Clinique + HRV"),
    ("XGBoost", "sél. RF", "Clinique + HRV"),
    ("SVR", "PCA 90 %", "Clinique + HRV"),
    # [PISTE] familles : ("Elastic Net", "toutes", "HRV temporel"), ("Elastic Net", "toutes", "HRV non linéaire")...
    # [PISTE] sélections : ("Elastic Net", "sél. Lasso", "Clinique + HRV"), ("LightGBM", "PCA 90 %", "Clinique + HRV")
    # [PISTE] idée du prof : des variables tirées des données brutes (fichiers RR), jointes à `donnees` en 1.5.
]
EXPERIENCES = [e for e in EXPERIENCES if JEUX[e[2]]]

## 3.1 Vérification rapide

Chaque expérience est ajustée une fois, sur le 1er pli, avant la longue évaluation : une erreur de configuration apparaît ici en quelques secondes.

In [ ]:
def nom_experience(exp):
    return " · ".join(exp)


def taches_plis(repetitions=REPETITIONS):
    """(répétition, pli, enfants d'entraînement, enfants de test), découpés exactement comme dans run_cv."""
    taches = []
    for r in sorted(plis_partages["repeat"].unique()):
        if repetitions is not None and r not in repetitions:
            continue
        fr = plis_partages[plis_partages["repeat"] == r]
        for k in sorted(fr["fold"].unique()):
            taches.append((r, k, fr.loc[fr["fold"] != k, "child_id"].tolist(), fr.loc[fr["fold"] == k, "child_id"].tolist()))
    return taches


def details_modele(pipe):
    """Hyperparamètres choisis, variables sélectionnées et, pour Elastic Net, coefficients."""
    modele, selection = pipe.named_steps["modele"], pipe.named_steps["selection"]
    noms = pipe.named_steps["prep"].get_feature_names_out()
    details = {}
    if isinstance(selection, SelectFromModel):
        noms = noms[selection.get_support()]
        details["selection"] = list(noms)
    elif isinstance(selection, PCA):
        noms = np.array([f"composante {i + 1}" for i in range(selection.n_components_)])
        details["composantes"] = selection.n_components_
    if isinstance(modele, RandomizedSearchCV):
        details["hyper"] = modele.best_params_
    elif isinstance(modele, ElasticNetCV):
        details["hyper"] = {"alpha": modele.alpha_, "l1_ratio": modele.l1_ratio_}
        details["coefs"] = pd.Series(modele.coef_, index=noms)
    return details


def ajuster_predire(exp, train_ids, test_ids, etiq):
    """Un pli : apprend sur train_ids, prédit test_ids (un modèle par cible)."""
    warnings.filterwarnings("ignore")
    modele, selection, jeu = exp
    colonnes = JEUX[jeu]
    predictions, details = {}, {}
    for cible in etiq.columns:
        pipe = faire_pipeline(modele, selection, colonnes).fit(X.loc[train_ids, colonnes], etiq.loc[train_ids, cible])
        predictions[cible] = np.ravel(pipe.predict(X.loc[test_ids, colonnes]))
        details[cible] = details_modele(pipe)
    return predictions, details


def essai(exp, tr, te):
    t = time.time()
    pred, _ = ajuster_predire(exp, tr, te, etiquettes[["QIT"]])
    return nom_experience(exp), mean_absolute_error(etiquettes.loc[te, "QIT"], pred["QIT"]), time.time() - t


_, _, tr0, te0 = taches_plis([1])[0]
essais = Parallel(n_jobs=N_JOBS)(delayed(essai)(exp, tr0, te0) for exp in EXPERIENCES)
display(pd.DataFrame(essais, columns=["expérience", "MAE (1er pli, QIT)", "secondes"]).set_index("expérience").round(2))

# 4. Evaluation

Protocole du jeudi : **10 répétitions × 5 plis** (`folds.csv`), une boucle interne pour les hyperparamètres, toutes les cibles. Les prédictions sont écrites au format de `evaluate_model.py` (`data/cleaned/predictions/`), donc comparables avec celles de l'autre groupe. Avec `RAPIDE = True`, seule la 1re répétition est faite : de quoi vérifier, pas conclure.

## 4.1 Lancement

In [ ]:
DOSSIER_PRED = DOSSIER_PARTAGE / "predictions"
DOSSIER_PRED.mkdir(parents=True, exist_ok=True)
REFERENCE = "QIT moyen (référence)"


def lancer(exp, fichier):
    """Ajuste l'expérience sur tous les plis en parallèle, puis écrit les prédictions avec run_cv (format commun)."""
    taches = taches_plis()
    sorties = Parallel(n_jobs=N_JOBS)(delayed(ajuster_predire)(exp, tr, te, etiquettes) for _, _, tr, te in taches)
    deja_calcule = {tuple(te): pred for (_, _, _, te), (pred, _) in zip(taches, sorties)}
    run_cv(lambda tr, te: deja_calcule[tuple(te)], fichier_plis, fichier_cohorte, fichier,
           targets=CIBLES, repeats=REPETITIONS, verbose=False)
    return [details for _, details in sorties]


fichiers, details_exp = {}, {}
debut = time.time()
for exp in EXPERIENCES:
    t = time.time()
    nom = nom_experience(exp)
    fichiers[nom] = DOSSIER_PRED / f"predictions_{re.sub(r'[^a-z0-9]+', '_', normaliser_texte(nom)).strip('_')}.csv"
    details_exp[nom] = lancer(exp, fichiers[nom])
    print(f"{nom} : {time.time() - t:.0f} s")
print(f"Total : {(time.time() - debut) / 60:.1f} min pour {len(EXPERIENCES)} expériences × {len(CIBLES)} cibles "
      f"× {len(taches_plis())} plis")

## 4.2 Comparaison des modèles : T × K mesures

Une mesure par pli de test, résumée par moyenne ± écart-type (MAE, MSE, RMSE, R²), puis écrite dans `results/model_comparison.csv` au format du plan. La référence prédit, dans chaque pli, le score moyen des enfants d'entraînement : c'est le plancher à battre. Le R² d'un pli d'environ 36 enfants est très bruité ; celui de 4.3, calculé sur toutes les prédictions, est plus fiable.

In [ ]:
def mesures(vrai, pred):
    mse = mean_squared_error(vrai, pred)
    return {"MAE": mean_absolute_error(vrai, pred), "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(vrai, pred)}


lignes = []
for cible in CIBLES:
    for r, k, tr, te in taches_plis():
        lignes.append({"model": REFERENCE, "target": cible,
                       **mesures(etiquettes.loc[te, cible], np.full(len(te), etiquettes.loc[tr, cible].mean()))})
    for nom, fichier in fichiers.items():
        p = pd.read_csv(fichier)
        lignes += [{"model": nom, "target": cible, **mesures(g[f"true_{cible}"], g[f"predicted_{cible}"])}
                   for _, g in p.groupby(["repeat", "fold"])]
par_pli = pd.DataFrame(lignes)

comparaison_modeles = par_pli.groupby(["model", "target"], sort=False)[["MAE", "MSE", "RMSE", "R2"]].agg(["mean", "std"])
comparaison_modeles.columns = [f"{m.lower()}_{s}" for m, s in comparaison_modeles.columns]
comparaison_modeles = comparaison_modeles.reset_index()
DOSSIER_RESULTATS.mkdir(parents=True, exist_ok=True)
comparaison_modeles.round(4).to_csv(DOSSIER_RESULTATS / "model_comparison.csv", index=False)

MEILLEURS = {c: comparaison_modeles[(comparaison_modeles.target == c) & (comparaison_modeles.model != REFERENCE)]
             .sort_values("mae_mean")["model"].iloc[0] for c in CIBLES}
tableau = comparaison_modeles.sort_values(["target", "mae_mean"], key=lambda s: s.map(CIBLES.index) if s.name == "target" else s)
display(pd.DataFrame({
    "MAE": [f"{m:.2f} ± {s:.2f}" for m, s in zip(tableau.mae_mean, tableau.mae_std)],
    "RMSE": [f"{m:.2f} ± {s:.2f}" for m, s in zip(tableau.rmse_mean, tableau.rmse_std)],
    "R²": [f"{m:.3f} ± {s:.3f}" for m, s in zip(tableau.r2_mean, tableau.r2_std)],
    "": ["meilleur" if MEILLEURS[t] == m else "" for t, m in zip(tableau.target, tableau.model)],
}, index=pd.MultiIndex.from_frame(tableau[["target", "model"]], names=["cible", "modèle"])))

# Graphiques du plan : MAE et R² par modèle et par cible, puis la distribution des MAE par pli pour le QIT.
modeles = [REFERENCE] + list(fichiers)
fig, axes = plt.subplots(1, 2, figsize=(13, 0.5 * len(modeles) * max(1, len(CIBLES) / 2) + 1.5), sharey=True)
hauteur = 0.8 / len(CIBLES)
for j, cible in enumerate(CIBLES):
    t = comparaison_modeles[comparaison_modeles.target == cible].set_index("model").loc[modeles]
    pos = np.arange(len(modeles)) + (j - (len(CIBLES) - 1) / 2) * hauteur
    axes[0].barh(pos, t.mae_mean, height=hauteur, xerr=t.mae_std, label=cible, capsize=2)
    axes[1].barh(pos, t.r2_mean, height=hauteur, xerr=t.r2_std, label=cible, capsize=2)
axes[0].set(yticks=range(len(modeles)), yticklabels=modeles, xlabel="MAE (points) — plus bas = mieux")
axes[0].invert_yaxis()
axes[1].axvline(0, color="k", lw=1)
axes[1].set(xlabel="R² — au-dessus de 0 = mieux que la moyenne")
axes[1].legend(title="cible", loc="lower right")
plt.tight_layout()
fig.savefig(DOSSIER_RESULTATS / "comparaison_modeles.png", dpi=150)
plt.show()

qit = par_pli[par_pli.target == "QIT"]
ordre = qit.groupby("model")["MAE"].mean().sort_values().index[::-1]
fig, ax = plt.subplots(figsize=(9, 0.45 * len(ordre) + 1.2))
ax.boxplot([qit.loc[qit.model == m, "MAE"] for m in ordre], tick_labels=list(ordre), orientation="horizontal")
ax.axvline(qit.loc[qit.model == REFERENCE, "MAE"].mean(), ls="--", color="grey", label=REFERENCE)
ax.set(xlabel="MAE par pli de test (points de QIT)", title="QIT : MAE pli par pli")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 4.3 Intervalles de confiance (`evaluate_model.py`)

Le rapport commun de l'équipe : MAE et R² sur toutes les prédictions hors pli, avec un intervalle de confiance à 95 % (bootstrap sur les enfants), et le gain par rapport à la moyenne. **Si l'intervalle du gain contient 0, l'amélioration n'est pas démontrée.** Les comparaisons montrées sont celles qui comptent : le meilleur modèle contre les autres, et l'apport de la HRV au modèle clinique.

In [ ]:
resume_equipe, comparaison = evaluate(fichier_plis, fichier_cohorte, [str(f) for f in fichiers.values()],
                                      names=list(fichiers), targets=CIBLES, verbose=False)


def ic(bas, haut, chiffres=2):
    return f"[{bas:.{chiffres}f} ; {haut:.{chiffres}f}]"


r = resume_equipe
display(pd.DataFrame({
    "MAE [IC 95 %]": [f"{m:.2f} {ic(a, b)}" for m, a, b in zip(r.MAE, r.MAE_ci_low, r.MAE_ci_high)],
    "R² [IC 95 %]": [f"{m:.3f} {ic(a, b, 3)}" for m, a, b in zip(r.R2, r.R2_ci_low, r.R2_ci_high)],
    "gain vs moyenne [IC 95 %]": ["" if m == "mean_baseline" else f"{g:+.2f} {ic(a, b)}"
                                  for m, g, a, b in zip(r.model, r.MAE_gain_vs_baseline, r.gain_ci_low, r.gain_ci_high)],
    "gain démontré": ["" if m == "mean_baseline" else ("oui" if a > 0 else "non") for m, a in zip(r.model, r.gain_ci_low)],
}, index=pd.MultiIndex.from_frame(r[["target", "model"]].replace({"mean_baseline": REFERENCE}), names=["cible", "modèle"])))

if comparaison is not None:
    cles = []
    for _, c in comparaison.iterrows():
        meilleur = MEILLEURS[c.target]
        hrv_vs_clinique = {c.model_a, c.model_b} == {"Elastic Net · toutes · Clinique seul", "Elastic Net · toutes · Clinique + HRV"}
        if meilleur in (c.model_a, c.model_b) or hrv_vs_clinique:
            a, b, d, bas, haut = (c.model_a, c.model_b, c.MAE_a_minus_b, c.ci_low, c.ci_high) if c.model_a == meilleur \
                else (c.model_b, c.model_a, -c.MAE_a_minus_b, -c.ci_high, -c.ci_low)
            cles.append({"cible": c.target, "comparaison": f"{a}  vs  {b}", "MAE 1er − MAE 2e [IC 95 %]": f"{d:+.2f} {ic(bas, haut)}",
                         "différence démontrée": "oui" if bas > 0 or haut < 0 else "non"})
    display(pd.DataFrame(cles).set_index(["cible", "comparaison"]))

## 4.4 Analyse d'erreurs du meilleur modèle (QIT)

Prédit contre observé et résidus : un point = un enfant, prédit quand il était dans le pli de test (1re répétition). **Graphiques individuels : à ne pas partager.** Un modèle prudent ramène toutes ses prédictions vers la moyenne ; il se trompe donc surtout pour les QIT bas, ceux qui intéressent le plus.

In [ ]:
MEILLEUR = MEILLEURS["QIT"]
exp_meilleur = next(e for e in EXPERIENCES if nom_experience(e) == MEILLEUR)
hors_pli = pd.read_csv(fichiers[MEILLEUR])
hors_pli = hors_pli[hors_pli["repeat"] == hors_pli["repeat"].min()]
vrai, pred = hors_pli["true_QIT"].to_numpy(), hors_pli["predicted_QIT"].to_numpy()
residus = vrai - pred

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
bornes = [min(vrai.min(), pred.min()) - 2, max(vrai.max(), pred.max()) + 2]
axes[0].scatter(vrai, pred, s=12, alpha=0.6)
axes[0].plot(bornes, bornes, "k--", lw=1, label="prédiction parfaite")
axes[0].axhline(vrai.mean(), color="grey", lw=1, label="QIT moyen")
axes[0].set(xlabel="QIT observé", ylabel="QIT prédit", title=MEILLEUR)
axes[0].legend()
axes[1].scatter(pred, residus, s=12, alpha=0.6)
axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="QIT prédit", ylabel="résidu (observé − prédit)", title="Résidus")
plt.tight_layout()
plt.show()

bas = vrai < 90
grandes = np.abs(residus) > 2 * residus.std()
print(f"MAE : {np.abs(residus[bas]).mean():.1f} points pour les QIT < 90 (n = {bas.sum()}), "
      f"{np.abs(residus[~bas]).mean():.1f} pour les QIT ≥ 90 (n = {(~bas).sum()})")
print(f"{grandes.sum()} enfants avec une erreur supérieure à 2 écarts-types, dont {(grandes & bas).sum()} avec un QIT < 90")

## 4.5 Test de permutation (QIT)

On mélange les QIT entre les enfants et on refait toute la procédure, réglages compris (1re répétition). Si le meilleur modèle ne fait pas nettement mieux qu'avec des QIT mélangés, sa performance ne se distingue pas du hasard. Avec `RAPIDE = True` (20 permutations), p ne peut pas descendre sous 0,048.

In [ ]:
taches_1 = taches_plis([1])
melanges = [pd.DataFrame({"QIT": np.random.default_rng(GRAINE + i).permutation(etiquettes["QIT"].to_numpy())},
                         index=etiquettes.index) for i in range(N_PERMUTATIONS)]
sorties = Parallel(n_jobs=N_JOBS)(delayed(ajuster_predire)(exp_meilleur, tr, te, em)
                                  for em in melanges for _, _, tr, te in taches_1)
mae_melange = np.array([
    mean_absolute_error(em["QIT"], pd.concat([pd.Series(sorties[i * len(taches_1) + j][0]["QIT"], index=te)
                                              for j, (_, _, _, te) in enumerate(taches_1)]).loc[em.index])
    for i, em in enumerate(melanges)])
mae_vraie = mean_absolute_error(vrai, pred)
p_valeur = (1 + np.sum(mae_melange <= mae_vraie)) / (1 + N_PERMUTATIONS)

fig, ax = plt.subplots(figsize=(6.5, 2.8))
ax.hist(mae_melange, bins=20, color="lightgrey", edgecolor="grey", label="QIT mélangés")
ax.axvline(mae_vraie, color="tab:red", lw=2, label=f"vrai modèle (p = {p_valeur:.3f})")
ax.set(xlabel="MAE (points de QIT)", ylabel="permutations", title=MEILLEUR)
ax.legend()
plt.tight_layout()
plt.show()

# 5. Prédiction conforme (QIT)

Un QIT prédit seul ne dit rien de sa marge d'erreur. La **prédiction conforme** donne un intervalle [ŷ − q ; ŷ + q] qui contient le vrai QIT avec une probabilité d'au moins 1 − α (90 %), quel que soit le modèle :

1. les erreurs du modèle sont mesurées par validation croisée sur les enfants d'entraînement ;
2. q est le quantile 1 − α de ces erreurs, avec une petite correction pour l'effectif ;
3. pour un nouvel enfant, l'intervalle est sa prédiction ± q.

La couverture est vérifiée sur les plis de test. La **largeur** dit si le modèle sert à quelque chose : si elle est à peine plus petite que celle de l'intervalle construit autour du QIT moyen, le modèle n'apporte presque rien.

In [ ]:
def quantile_conforme(erreurs, alpha=ALPHA):
    n = len(erreurs)
    return np.quantile(erreurs, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")


def intervalle_un_pli(exp, train_ids, test_ids):
    etiq = etiquettes[["QIT"]]
    erreurs = []
    for tr_i, va_i in KFold(K_PLIS, shuffle=True, random_state=0).split(train_ids):
        tr_ids, va_ids = [train_ids[i] for i in tr_i], [train_ids[i] for i in va_i]
        p, _ = ajuster_predire(exp, tr_ids, va_ids, etiq)
        erreurs.extend(np.abs(etiq.loc[va_ids, "QIT"].to_numpy() - p["QIT"]))
    p, _ = ajuster_predire(exp, train_ids, test_ids, etiq)
    y_train = etiq.loc[train_ids, "QIT"]
    return pd.DataFrame({"vrai": etiq.loc[test_ids, "QIT"].to_numpy(), "pred": p["QIT"],
                         "q": quantile_conforme(np.array(erreurs)), "moyenne": y_train.mean(),
                         "q_ref": quantile_conforme(np.abs(y_train - y_train.mean()).to_numpy())}, index=test_ids)


conforme = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(intervalle_un_pli)(exp_meilleur, tr, te) for _, _, tr, te in taches_1))
couverture = (np.abs(conforme.vrai - conforme.pred) <= conforme.q).mean()
couverture_ref = (np.abs(conforme.vrai - conforme.moyenne) <= conforme.q_ref).mean()
q_final = quantile_conforme(np.abs(residus))     # pour un nouvel enfant : erreurs hors pli de tous les enfants
display(pd.DataFrame({"couverture observée": [f"{couverture:.0%}", f"{couverture_ref:.0%}"],
                      "demi-largeur (points de QIT)": [f"± {conforme.q.mean():.1f}", f"± {conforme.q_ref.mean():.1f}"]},
                     index=[MEILLEUR, "autour du QIT moyen"]))
print(f"Pour un nouvel enfant : QIT prédit ± {q_final:.1f} points, avec une probabilité d'au moins {1 - ALPHA:.0%} "
      "de contenir son vrai QIT.")

# Graphique individuel (un trait = un enfant) : à ne pas partager.
tri = conforme.sort_values("pred")
dedans = np.abs(tri.vrai - tri.pred) <= tri.q
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.vlines(range(len(tri)), tri.pred - tri.q, tri.pred + tri.q, color="lightsteelblue", lw=2)
ax.scatter(range(len(tri)), tri.vrai, s=10, c=np.where(dedans, "black", "tab:red"), zorder=3)
ax.axhline(90, color="grey", ls=":", lw=1)
ax.set(xlabel="enfants triés par QIT prédit", ylabel="QIT", title=f"Intervalles à {1 - ALPHA:.0%} (rouge : vrai QIT en dehors)")
plt.tight_layout()
plt.show()

# 6. Explication (QIT)

- **Quelles variables le modèle utilise-t-il ?** Importance par permutation : on mélange une variable chez les enfants de test et on mesure de combien la MAE augmente. Elle est calculée sur des enfants non vus, par variable d'origine, donc valable pour tous les modèles, même après une PCA.
- **Ces choix sont-ils stables ?** Une variable importante dans un seul pli relève du bruit ; une variable importante dans presque tous les plis est plus crédible.

In [ ]:
def importance_un_pli(exp, train_ids, test_ids):
    warnings.filterwarnings("ignore")
    modele, selection, jeu = exp
    colonnes = JEUX[jeu]
    pipe = faire_pipeline(modele, selection, colonnes).fit(X.loc[train_ids, colonnes], etiquettes.loc[train_ids, "QIT"])
    r = permutation_importance(pipe, X.loc[test_ids, colonnes], etiquettes.loc[test_ids, "QIT"],
                               scoring="neg_mean_absolute_error", n_repeats=5, random_state=0)
    return pd.Series(r.importances_mean, index=colonnes)


importances = pd.concat(Parallel(n_jobs=N_JOBS)(delayed(importance_un_pli)(exp_meilleur, tr, te)
                                                for _, _, tr, te in taches_1), axis=1)
resume_imp = pd.DataFrame({"hausse de MAE (points)": importances.mean(axis=1),
                           "plis où elle compte": (importances > 0).mean(axis=1)}).sort_values("hausse de MAE (points)", ascending=False)
haut = resume_imp.head(15)[::-1]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.barh(haut.index, haut["hausse de MAE (points)"], color=np.where(haut["plis où elle compte"] >= 0.8, "tab:blue", "lightsteelblue"))
ax.axvline(0, color="k", lw=0.8)
ax.set(xlabel="hausse de la MAE quand la variable est mélangée (points de QIT)",
       title=f"{MEILLEUR}\nfoncé : utile dans au moins 80 % des plis")
plt.tight_layout()
plt.show()

details = [d["QIT"] for d in details_exp[MEILLEUR]]
if any("coefs" in d for d in details):
    coefs = pd.DataFrame([d["coefs"] for d in details])
    frequence = (coefs.fillna(0) != 0).mean()
    print(f"Elastic Net : {(frequence >= 0.8).sum()} variables retenues dans au moins 80 % des plis.")
if any("selection" in d for d in details):
    frequence = pd.Series([v for d in details for v in d["selection"]]).value_counts() / len(details)
    print(f"Sélection : {(frequence >= 0.8).sum()} variables gardées dans au moins 80 % des plis, "
          f"dont {list(frequence.index[:5])}")
if any("composantes" in d for d in details):
    print(f"PCA : {min(d['composantes'] for d in details)} à {max(d['composantes'] for d in details)} composantes.")
hyper = pd.DataFrame([d["hyper"] for d in details if "hyper" in d])
if "alpha" in hyper:
    print(f"Réglages : l1_ratio {hyper['l1_ratio'].value_counts().sort_index().to_dict()}, alpha médian {hyper['alpha'].median():.3g}")
elif len(hyper):
    print("Réglages les plus choisis :", hyper.astype(str).apply(lambda r: ", ".join(f"{k}={v}" for k, v in r.items()), axis=1)
          .value_counts().head(3).to_dict())
# [PISTE] Un réglage toujours au bord de son espace : élargir l'espace de ce côté (section 3).

**Ce que l'on peut conclure, et ce que l'on ne peut pas**

- Une variable importante est **associée** au QIT dans cette cohorte ; elle n'en est pas pour autant la **cause**. La HRV néonatale est un marqueur de la maturation du système nerveux autonome, pas un levier.
- « Que faire pour un enfant prédit sous 90 ? » est une question **causale**, à laquelle un modèle prédictif ne répond pas. Il peut au mieux **repérer** des enfants à suivre de plus près, si ses prédictions sont fiables (sections 4 et 5). Les interventions relèvent d'études cliniques.
- Avec un signal faible, les importances varient d'un pli à l'autre : une variable isolée ne doit pas être surinterprétée.

# 7. Résumé pour le tuteur

Généré à partir des résultats ci-dessus (chiffres agrégés seulement) et enregistré dans `results/SUMMARY.md`, comme prévu le vendredi. Le meilleur modèle de chaque cible est aussi réajusté sur tous les enfants et sauvegardé localement (`data/cleaned/modeles/`, prétraitement compris).

In [ ]:
def ligne_resultat(cible):
    m = MEILLEURS[cible]
    c = comparaison_modeles[(comparaison_modeles.target == cible) & (comparaison_modeles.model == m)].iloc[0]
    ref = comparaison_modeles[(comparaison_modeles.target == cible) & (comparaison_modeles.model == REFERENCE)].iloc[0]
    e = resume_equipe[(resume_equipe.target == cible) & (resume_equipe.model == m)].iloc[0]
    demontre = "démontré" if e.gain_ci_low > 0 else "non démontré"
    return (f"| {cible} | {m} | {c.mae_mean:.2f} ± {c.mae_std:.2f} | {c.r2_mean:.3f} | {ref.mae_mean:.2f} | "
            f"{e.MAE_gain_vs_baseline:+.2f} {ic(e.gain_ci_low, e.gain_ci_high)} ({demontre}) |")


qit_seul = par_pli[(par_pli.target == "QIT")]
apport = None
if {"Elastic Net · toutes · Clinique seul", "Elastic Net · toutes · Clinique + HRV"} <= set(qit_seul.model):
    apport = (qit_seul[qit_seul.model == "Elastic Net · toutes · Clinique seul"]["MAE"].mean()
              - qit_seul[qit_seul.model == "Elastic Net · toutes · Clinique + HRV"]["MAE"].mean())

SUMMARY = "\n".join([
    "# Prédiction du QIT à partir de la HRV néonatale : résumé",
    f"*Groupe A · Jade · généré le {date.today():%d/%m/%Y} par `01_prediction_qit.ipynb`"
    + (" · mode rapide (1 répétition), chiffres provisoires*" if RAPIDE else f" · {T_REPETITIONS} × {K_PLIS} plis*"),
    "",
    "## Données et méthode",
    f"- {len(X)} enfants de la cohorte AuBE avec un enregistrement M0 et des scores WPPSI-III complets ; "
    f"{len(cols_lin)} indices HRV linéaires, {len(cols_nl)} non linéaires, {len(cols_clin)} covariables cliniques.",
    f"- Modèles : {', '.join(sorted({e[0] for e in EXPERIENCES}))} ; sélection de variables dans les plis "
    "(forêt, PCA 90 %) ; hyperparamètres réglés par validation croisée interne.",
    f"- Évaluation : plis communs de l'équipe (`evaluate_model.py`), cibles {', '.join(CIBLES)}.",
    "",
    "## Résultats (meilleur modèle par cible)",
    "| Cible | Modèle | MAE ± écart-type | R² | MAE de la moyenne | Gain vs moyenne [IC 95 %] |",
    "|---|---|---|---|---|---|",
    *[ligne_resultat(c) for c in CIBLES],
    "",
    f"- QIT : test de permutation p = {p_valeur:.3f} ; intervalle conforme à {1 - ALPHA:.0%} : ± {q_final:.1f} points "
    f"(couverture observée {couverture:.0%}, contre ± {conforme.q_ref.mean():.1f} autour du QIT moyen).",
    *([f"- Apport de la HRV au modèle clinique (Elastic Net, QIT) : MAE {-apport:+.2f} points."] if apport is not None else []),
    "- Référence : rapport de Samuel Fornet, MAE ≈ 10,3 et R² ≤ 0 en validation croisée imbriquée.",
    "",
    "## Limites",
    "- Petit effectif (≈ 180 enfants) pour ≈ 200 variables ; pas de cohorte externe de validation.",
    "- Signal faible : une association n'est pas une cause, et la prédiction individuelle reste imprécise.",
    "",
    "## Suite",
    "- Variables tirées des données brutes (fichiers RR, signal ECG avec le groupe B) ; familles d'indices séparées.",
    "- Elastic Net bayésien / processus gaussien ; classification QIT < 90 (AUC, kappa).",
])
(DOSSIER_RESULTATS / "SUMMARY.md").write_text(SUMMARY + "\n", encoding="utf-8")
display(Markdown(SUMMARY))

# Meilleur modèle de chaque cible, réajusté sur tous les enfants (local, jamais commité).
(DOSSIER_PARTAGE / "modeles").mkdir(exist_ok=True)
for cible in CIBLES:
    modele, selection, jeu = next(e for e in EXPERIENCES if nom_experience(e) == MEILLEURS[cible])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipe = faire_pipeline(modele, selection, JEUX[jeu]).fit(X[JEUX[jeu]], etiquettes[cible])
    joblib.dump(pipe, DOSSIER_PARTAGE / "modeles" / f"{cible}.joblib")
print(f"Modèles sauvegardés dans {DOSSIER_PARTAGE / 'modeles'}")

# 8. Avant de committer

Le plan demande de vérifier qu'aucune donnée de santé n'est suivie par git. `results/` ne contient que des chiffres agrégés ; `data/`, les CSV et les modèles restent locaux.

In [ ]:
suivis = subprocess.run(["git", "ls-files"], cwd=RACINE, capture_output=True, text=True).stdout.splitlines()
suspects = [f for f in suivis if re.search(r"\.(csv|xlsx?|mat|edf|zip|pkl|joblib)$", f, re.I)]
print("Fichiers de données suivis par git :", suspects or "aucun")
print("Rappel : les sorties de ce notebook sont vidées au commit par nbstripout ; sans ce filtre, "
      "faire « Restart & Clear Output » avant git add.")

# 9. Suite

- **Réglages à tester** (marqués `[PISTE]`, toujours jugés sur la section 4) : `TOP_K`, `VARIANCE_PCA`, `SEUIL_CORR`, `PERIODES`, espaces d'hyperparamètres.
- **Données brutes** : variables calculées sur les fichiers RR (idée du prof), jointes en 1.5 et ajoutées comme jeu de variables.
- **Explication plus fine des arbres** : valeurs SHAP (`shap.TreeExplainer` sur LightGBM ou XGBoost).
- **Classification** : QIT < 90 contre ≥ 90, avec AUC et kappa de Cohen (l'exactitude seule est trompeuse avec 78 % d'enfants au-dessus de 90).